# 优化算法 (Optimization Algorithms)

> 参考：[动手学深度学习 v2 第11章](https://zh-v2.d2l.ai/chapter_optimization/index.html)

**核心问题**：最小化训练损失（经验风险），同时希望泛化误差（真实风险）也尽可能小。

## 目录
1. [优化与深度学习](#1-优化与深度学习)
2. [凸性](#2-凸性)
3. [梯度下降](#3-梯度下降)
4. [随机梯度下降（SGD）](#4-随机梯度下降sgd)
5. [小批量随机梯度下降](#5-小批量随机梯度下降)
6. [动量法（Momentum）](#6-动量法momentum)
7. [AdaGrad](#7-adagrad)
8. [RMSProp](#8-rmsprop)
9. [Adadelta](#9-adadelta)
10. [Adam](#10-adam)
11. [学习率调度器](#11-学习率调度器)
12. [综合对比与实验](#12-综合对比与实验)


## 0. 环境配置

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import math

# !pip install d2l
import d2l.torch as d2l

device = d2l.try_gpu()
print(f"使用设备：{device}")
print(f"PyTorch 版本：{torch.__version__}")


## 1. 优化与深度学习

### 1.1 目标函数差异

| 概念 | 说明 |
|---|---|
| 风险（Risk） | 期望损失，目标是最小化它 |
| 经验风险（Empirical Risk） | 训练集上的平均损失，实际可计算 |
| 优化目标 | 最小化经验风险，以期减小风险 |

> 两者差异 = **泛化误差**。优化只管训练集，不保证泛化。

### 1.2 三大挑战

**① 局部极小值（Local Minima）**：非凸损失面存在大量局部最优解。

**② 鞍点（Saddle Points）**：梯度为零但非极值点。在高维空间中，鞍点比局部极小值更普遍。
- 海森矩阵特征值全正 → 局部极小值
- 海森矩阵特征值全负 → 局部极大值
- **混合** → 鞍点（高维中概率极大）

**③ 梯度消失（Vanishing Gradients）**：激活函数饱和区梯度趋近 0，优化停滞。


In [ ]:
# ── 可视化三大挑战 ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. 局部极小值
x = np.linspace(-2.5, 2.5, 300)
y1 = x**4 - 4*x**2 + 0.5*x
axes[0].plot(x, y1, 'b-', linewidth=2)
axes[0].scatter([-1.35], [y1[np.argmin(np.abs(x+1.35))]], s=80, c='red', zorder=5, label='局部极小值')
axes[0].scatter([1.4],  [y1[np.argmin(np.abs(x-1.4))]],  s=80, c='green', zorder=5, label='全局极小值')
axes[0].set_title('局部极小值 vs 全局极小值'); axes[0].legend(); axes[0].set_xlabel('x')

# 2. 鞍点（f(x,y) = x^2 - y^2，在x方向取切面）
x = np.linspace(-2, 2, 300)
y2 = x**3
axes[1].plot(x, y2, 'b-', linewidth=2)
axes[1].scatter([0], [0], s=120, c='red', zorder=5, label='鞍点（梯度=0）')
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('鞍点（f(x)=x³）'); axes[1].legend(); axes[1].set_xlabel('x')

# 3. 梯度消失（tanh 在饱和区梯度趋零）
x = np.linspace(-6, 6, 300)
tanh_grad = 1 - np.tanh(x)**2
axes[2].plot(x, np.tanh(x), label='tanh(x)')
axes[2].plot(x, tanh_grad,  label="tanh'(x)")
axes[2].fill_between(x, tanh_grad, alpha=0.2)
axes[2].axvline(4, color='red', linestyle='--', alpha=0.7, label=f"x=4, grad≈{1-np.tanh(4)**2:.4f}")
axes[2].set_title('梯度消失（tanh 饱和区）'); axes[2].legend(); axes[2].set_xlabel('x')

plt.tight_layout(); plt.show()


## 2. 凸性

### 2.1 凸集与凸函数

**凸集**：对任意 $a, b \in \mathcal{X}$ 和 $\lambda \in [0,1]$，满足 $\lambda a + (1-\lambda)b \in \mathcal{X}$。

**凸函数**：定义域为凸集，且满足：
$$\lambda f(x) + (1-\lambda)f(x') \geq f(\lambda x + (1-\lambda)x'), \quad \forall \lambda \in [0,1]$$

**核心性质**：凸函数的任意局部极小值 = 全局极小值。

### 2.2 Jensen 不等式

$$E[f(X)] \geq f(E[X])$$

直觉：凸函数的期望 $\geq$ 期望的凸函数。

### 2.3 二阶条件

- 一维：$f''(x) \geq 0$（曲率非负）
- 多维：海森矩阵 $\mathbf{H} \succeq 0$（半正定），即 $\mathbf{x}^\top \mathbf{H} \mathbf{x} \geq 0$

### 2.4 约束优化与拉格朗日函数

对约束 $c_i(\mathbf{x}) \leq 0$，构造拉格朗日函数：
$$L(\mathbf{x}, \alpha_1, \ldots, \alpha_n) = f(\mathbf{x}) + \sum_i \alpha_i c_i(\mathbf{x}), \quad \alpha_i \geq 0$$

常用简化：添加 $\frac{\lambda}{2}\|\mathbf{w}\|^2$ 惩罚项（权重衰减/L2正则化）。


In [ ]:
# ── 凸函数与非凸函数可视化 ──
x = np.linspace(-3, 3, 300)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 凸函数：x^2
axes[0].plot(x, x**2, 'b-', linewidth=2)
xa, xb = -2, 1.5
axes[0].plot([xa, xb], [xa**2, xb**2], 'r--o', linewidth=2, label='弦（位于曲线上方）')
axes[0].set_title('凸函数：$f(x)=x^2$'); axes[0].legend()

# 非凸函数：sin(x)
axes[1].plot(x, np.sin(x), 'b-', linewidth=2)
xa, xb = -1, 2.5
lam = 0.5
xmid = lam*xa + (1-lam)*xb
axes[1].plot([xa, xb], [np.sin(xa), np.sin(xb)], 'r--o', linewidth=2, label='弦（穿过曲线）')
axes[1].set_title('非凸函数：$f(x)=\sin(x)$'); axes[1].legend()

# Jensen不等式示意
x_pts = np.array([-2, -1, 0, 1, 2], dtype=float)
p = np.array([0.1, 0.2, 0.4, 0.2, 0.1])
f_pts = x_pts**2
ex = np.sum(p * x_pts)
efx = np.sum(p * f_pts)
fex = ex**2
axes[2].plot(x, x**2, 'b-', linewidth=2, label='$f(x)=x^2$')
axes[2].scatter([ex], [fex], s=150, c='green', zorder=5, label=f'$f(E[X])={fex:.2f}$')
axes[2].scatter([ex], [efx], s=150, c='red', marker='^', zorder=5, label=f'$E[f(X)]={efx:.2f}$')
axes[2].set_title('Jensen 不等式：$E[f(X)] \geq f(E[X])$'); axes[2].legend()

plt.tight_layout(); plt.show()


## 3. 梯度下降

### 3.1 核心更新规则

从泰勒展开出发：$f(\mathbf{x} + \boldsymbol{\epsilon}) \approx f(\mathbf{x}) + \boldsymbol{\epsilon}^\top \nabla f(\mathbf{x})$，令 $\boldsymbol{\epsilon} = -\eta \nabla f(\mathbf{x})$ 可保证函数值下降：

$$\mathbf{x} \leftarrow \mathbf{x} - \eta \nabla f(\mathbf{x})$$

其中 $\eta > 0$ 为**学习率**。

### 3.2 学习率的影响

| $\eta$ | 结果 |
|---|---|
| 过小（如 0.05） | 收敛极慢 |
| 适中（如 0.2）  | 平滑收敛 |
| 过大（如 1.1）  | 发散 |

### 3.3 牛顿法（二阶方法）

$$\mathbf{x} \leftarrow \mathbf{x} - \eta \mathbf{H}^{-1} \nabla f(\mathbf{x})$$

- 优点：近优处**二次收敛速度**
- 缺点：计算 $\mathbf{H}^{-1}$ 代价 $O(d^3)$，深度学习中不可行

实践替代：**预处理对角近似**：$\mathbf{x} \leftarrow \mathbf{x} - \eta \cdot \mathrm{diag}(\mathbf{H})^{-1} \nabla f(\mathbf{x})$


In [ ]:
# ── 学习率对梯度下降的影响 ──
def f(x):  return x**2 / 2 + 0.2*x

def gd(eta, n_steps=20):
    x = 10.0
    history = [x]
    for _ in range(n_steps):
        x -= eta * (x + 0.2)   # f'(x) = x + 0.2
        history.append(x)
    return history

x_range = np.linspace(-2, 11, 300)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, eta in zip(axes, [0.05, 0.4, 1.1]):
    hist = gd(eta, n_steps=30)
    ax.plot(x_range, f(np.array(x_range)), 'b-', linewidth=1.5, alpha=0.6)
    ax.plot(hist, [f(x) for x in hist], 'r-o', markersize=4)
    ax.set_title(f'$\eta={eta}$  最终 x={hist[-1]:.3f}')
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')
plt.suptitle('学习率对梯度下降收敛的影响', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# ── 多维梯度下降：病态曲率示例 ──
# f(x1, x2) = 0.1*x1^2 + 2*x2^2  — x2 方向曲率大 10x
def f2d(x1, x2): return 0.1*x1**2 + 2*x2**2
def grad_f2d(x1, x2): return np.array([0.2*x1, 4*x2])

def gd_2d(eta, n_steps=20, x0=(5.0, -2.0)):
    x = np.array(x0, dtype=float)
    hist = [x.copy()]
    for _ in range(n_steps):
        x -= eta * grad_f2d(*x)
        hist.append(x.copy())
    return np.array(hist)

x1_range = np.linspace(-6, 6, 200)
x2_range = np.linspace(-3, 3, 200)
X1, X2 = np.meshgrid(x1_range, x2_range)
Z = f2d(X1, X2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (eta, label) in zip(axes, [(0.4, 'η=0.4（适中）'), (0.7, 'η=0.7（过大→振荡）')]):
    hist = gd_2d(eta)
    ax.contour(X1, X2, Z, levels=30, cmap='Blues')
    ax.plot(hist[:, 0], hist[:, 1], 'r-o', markersize=4)
    ax.scatter(*hist[0], c='green', s=100, zorder=5, label='起点')
    ax.scatter(0, 0, c='gold', s=100, marker='*', zorder=5, label='最优')
    ax.set_title(label); ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.legend()
plt.suptitle('病态曲率下梯度下降', fontsize=13)
plt.tight_layout(); plt.show()


## 4. 随机梯度下降（SGD）

### 4.1 核心思想

目标函数为均值：$f(\mathbf{x}) = \frac{1}{n}\sum_{i=1}^n f_i(\mathbf{x})$

**全批量梯度下降**：$\mathbf{x} \leftarrow \mathbf{x} - \eta \nabla f(\mathbf{x})$，计算代价 $O(n)$

**SGD**：随机抽取一个样本 $i$：
$$\mathbf{x} \leftarrow \mathbf{x} - \eta \nabla f_i(\mathbf{x})$$

无偏性：$\mathbb{E}_i[\nabla f_i(\mathbf{x})] = \nabla f(\mathbf{x})$

### 4.2 学习率衰减策略

随着迭代推进，噪声梯度的**方差**需要被压制：

| 策略 | 公式 |
|---|---|
| 分段常数 | $\eta(t) = \eta_i$（分段） |
| 指数衰减 | $\eta(t) = \eta_0 e^{-\lambda t}$ |
| 多项式衰减 | $\eta(t) = \eta_0 (\beta t+1)^{-\alpha}$ |

**收敛速率**（凸情形）：$O(1/\sqrt{T})$


In [ ]:
# ── 学习率衰减策略对比 ──
T = 100
t = np.arange(T)

schedules = {
    '常数 η=0.2':       np.ones(T) * 0.2,
    '指数衰减 λ=0.05': 0.5 * np.exp(-0.05 * t),
    '多项式衰减 α=0.5': 0.5 * (0.1 * t + 1) ** (-0.5),
    '分段常数':          np.where(t < 30, 0.5, np.where(t < 60, 0.1, 0.02)),
}

plt.figure(figsize=(8, 4))
for label, lr in schedules.items():
    plt.plot(t, lr, label=label)
plt.xlabel('迭代步数'); plt.ylabel('学习率')
plt.title('学习率衰减策略对比')
plt.legend(); plt.tight_layout(); plt.show()


## 5. 小批量随机梯度下降

### 5.1 核心公式

从批量 $\mathcal{B}_t$（大小 $b$）计算梯度：

$$\mathbf{g}_t = \frac{1}{|\mathcal{B}_t|} \sum_{i \in \mathcal{B}_t} \nabla f_i(\mathbf{x})$$

$$\mathbf{x}_t \leftarrow \mathbf{x}_{t-1} - \eta_t \mathbf{g}_t$$

**方差缩减**：相比单样本 SGD，方差降低 $b^{-1/2}$ 倍。

### 5.2 批量大小 vs 计算效率

| 模式 | 性能 |
|---|---|
| 逐元素 | ~0.016 Gflops（极慢） |
| 列向量化 | ~9–88 Gflops |
| 矩阵乘法 | ~84–555 Gflops |

> 向量化 = 充分利用 GPU/CPU 的 SIMD 指令集。

### 5.3 批量大小与学习率的关系

批量越大 → 梯度方差越小 → 可使用更大学习率（线性缩放规则：$\eta \propto b$）。


In [ ]:
# ── PyTorch Mini-batch SGD 标准用法 ──
torch.manual_seed(42)
n, d = 1000, 5
X_data = torch.randn(n, d)
true_w = torch.randn(d)
y_data = X_data @ true_w + 0.1 * torch.randn(n)

dataset = torch.utils.data.TensorDataset(X_data, y_data)

net = nn.Sequential(nn.Linear(d, 1))
loss_fn = nn.MSELoss()

results = {}
for batch_size in [1, 32, 256, n]:
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    model = nn.Sequential(nn.Linear(d, 1))
    # 学习率按批量大小线性缩放
    lr = 0.01 * (batch_size / 32)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    losses = []
    for epoch in range(15):
        epoch_loss = 0
        for Xb, yb in loader:
            optimizer.zero_grad()
            l = loss_fn(model(Xb).squeeze(), yb)
            l.backward()
            optimizer.step()
            epoch_loss += l.item()
        losses.append(epoch_loss / len(loader))
    results[f'bs={batch_size}, lr={lr:.4f}'] = losses

plt.figure(figsize=(8, 4))
for label, loss_curve in results.items():
    plt.plot(loss_curve, label=label)
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('不同批量大小的收敛曲线'); plt.legend(); plt.tight_layout(); plt.show()


## 6. 动量法（Momentum）

### 6.1 核心思想

用**梯度的指数加权平均**（泄漏平均）替代瞬时梯度，抑制振荡方向、加速收敛方向：

$$\mathbf{v}_t \leftarrow \beta \mathbf{v}_{t-1} + \mathbf{g}_{t}$$

$$\mathbf{x}_t \leftarrow \mathbf{x}_{t-1} - \eta_t \mathbf{v}_t$$

递归展开：$\mathbf{v}_t = \sum_{\tau=0}^{t-1} \beta^\tau \mathbf{g}_{t-\tau}$（指数加权历史梯度）

**有效学习率**：等比数列求和 $= \frac{\eta}{1-\beta}$；有效窗口约 $\frac{1}{1-\beta}$ 步。

| $\beta$ | 有效窗口 | 行为 |
|---|---|---|
| 0.5 | 2 步 | 噪声较大 |
| 0.9 | 10 步 | 标准设置 |
| 0.95 | 20 步 | 更平滑 |

### 6.2 PyTorch API

```python
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9)
```


In [ ]:
# ── 动量法 vs 梯度下降（病态曲率）──
def f2d(x1, x2):    return 0.1*x1**2 + 2*x2**2
def grad_f2d(x1, x2): return np.array([0.2*x1, 4*x2])

def momentum_2d(eta, beta, n_steps=30, x0=(5.0, -2.0)):
    x = np.array(x0, dtype=float)
    v = np.zeros(2)
    hist = [x.copy()]
    for _ in range(n_steps):
        g = grad_f2d(*x)
        v = beta * v + g
        x -= eta * v
        hist.append(x.copy())
    return np.array(hist)

x1_r = np.linspace(-6, 6, 200)
x2_r = np.linspace(-3, 3, 200)
X1, X2 = np.meshgrid(x1_r, x2_r)
Z = f2d(X1, X2)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
configs = [
    ('GD η=0.4',        lambda: momentum_2d(0.4, 0.0)),
    ('动量 η=0.4 β=0.9', lambda: momentum_2d(0.4, 0.9)),
    ('动量 η=0.6 β=0.5', lambda: momentum_2d(0.6, 0.5)),
]
for ax, (title, fn) in zip(axes, configs):
    hist = fn()
    ax.contour(X1, X2, Z, levels=25, cmap='Blues', alpha=0.6)
    ax.plot(hist[:, 0], hist[:, 1], 'r-o', markersize=4)
    ax.scatter(*hist[0],  c='green', s=100, zorder=5)
    ax.scatter(0, 0, c='gold', s=150, marker='*', zorder=5)
    ax.set_title(title); ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
plt.suptitle('动量法减弱病态曲率下的振荡', fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# ── PyTorch Momentum API ──
# 使用 d2l 的 Fashion-MNIST 训练工具对比
def get_data():
    return d2l.load_data_fashion_mnist(batch_size=256)

# 构建简单 LeNet
def get_net():
    net = nn.Sequential(
        nn.Conv2d(1, 6, kernel_size=5, padding=2), nn.Sigmoid(),
        nn.AvgPool2d(kernel_size=2, stride=2),
        nn.Conv2d(6, 16, kernel_size=5), nn.Sigmoid(),
        nn.AvgPool2d(kernel_size=2, stride=2),
        nn.Flatten(),
        nn.Linear(16*5*5, 120), nn.Sigmoid(),
        nn.Linear(120, 84), nn.Sigmoid(),
        nn.Linear(84, 10))
    return net

train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size=256)

# SGD without momentum
net1 = get_net()
optimizer1 = torch.optim.SGD(net1.parameters(), lr=0.05)
# SGD with momentum
net2 = get_net()
optimizer2 = torch.optim.SGD(net2.parameters(), lr=0.05, momentum=0.9)

print("training SGD (no momentum)...")
d2l.train_ch6(net1, train_iter, test_iter, 10, 0.05, device)


## 7. AdaGrad

### 7.1 核心思想

对**每个参数维度**维护历史梯度平方累计，为不频繁更新的参数提供更大的有效学习率（适合稀疏特征）：

$$\mathbf{s}_t \leftarrow \mathbf{s}_{t-1} + \mathbf{g}_t^2 \quad \text{（逐元素平方累加）}$$

$$\mathbf{x}_t \leftarrow \mathbf{x}_{t-1} - \frac{\eta}{\sqrt{\mathbf{s}_t + \epsilon}} \odot \mathbf{g}_t$$

- $\mathbf{s}_0 = \mathbf{0}$，$\epsilon \approx 10^{-7}$（防零除）
- 有效学习率近似 $O(t^{-1/2})$，随时间单调递减

### 7.2 优缺点

| 优点 | 缺点 |
|---|---|
| 自适应每维度学习率 | 学习率单调递减，可能过早停止 |
| 适合 NLP 稀疏特征 | 长期训练后学习率趋于零 |
| 无需手动调 LR 衰减 | 不适合非平稳（non-stationary）目标 |

### 7.3 PyTorch API

```python
optimizer = torch.optim.Adagrad(params, lr=0.1)
```


In [ ]:
# ── 可视化 AdaGrad 的逐维度自适应学习率 ──
np.random.seed(42)
T = 200
# 模拟：x1 经常更新，x2 稀疏更新
g1 = np.random.randn(T) * 1.0   # 频繁梯度（大方差）
g2 = np.random.randn(T) * 0.1   # 稀疏梯度（小方差）
g2[np.random.rand(T) > 0.2] = 0  # 80% 时刻梯度为 0

s1, s2 = 0, 0
eps = 1e-7
eff_lr1, eff_lr2 = [], []
for t in range(T):
    s1 += g1[t]**2
    s2 += g2[t]**2
    eff_lr1.append(0.1 / np.sqrt(s1 + eps))
    eff_lr2.append(0.1 / np.sqrt(s2 + eps))

plt.figure(figsize=(9, 4))
plt.plot(eff_lr1, label='频繁维度（大梯度）→ LR 快速衰减')
plt.plot(eff_lr2, label='稀疏维度（小梯度）→ LR 缓慢衰减')
plt.xlabel('迭代步数'); plt.ylabel('有效学习率')
plt.title('AdaGrad 逐维度自适应学习率')
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── PyTorch Adagrad ──
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size=256)
net = get_net()
optimizer = torch.optim.Adagrad(net.parameters(), lr=0.1)
d2l.train_ch6(net, train_iter, test_iter, num_epochs=10, lr=0.1, device=device)


## 8. RMSProp

### 8.1 核心思想

修复 AdaGrad 学习率单调递减的问题：用**指数加权移动平均（EMA）**替代无界累加：

$$\mathbf{s}_t \leftarrow \gamma \mathbf{s}_{t-1} + (1-\gamma) \mathbf{g}_t^2$$

$$\mathbf{x}_t \leftarrow \mathbf{x}_{t-1} - \frac{\eta}{\sqrt{\mathbf{s}_t + \epsilon}} \odot \mathbf{g}_t$$

- $\gamma = 0.9$（默认），有效窗口 $\approx \frac{1}{1-\gamma} = 10$ 步
- $\epsilon \approx 10^{-6}$

**与 AdaGrad 对比**：

| | AdaGrad | RMSProp |
|---|---|---|
| 状态更新 | $\mathbf{s}_t = \mathbf{s}_{t-1} + \mathbf{g}_t^2$（无界） | $\mathbf{s}_t = \gamma\mathbf{s}_{t-1} + (1-\gamma)\mathbf{g}_t^2$（有界） |
| LR 趋势 | 单调递减 | 受控（由 $\gamma$ 调节） |
| 适合场景 | 凸问题 | 深度学习（非凸） |

### 8.2 PyTorch API

```python
# PyTorch 中 gamma 参数名为 alpha
optimizer = torch.optim.RMSprop(params, lr=0.01, alpha=0.9)
```


In [ ]:
# ── RMSProp vs AdaGrad：状态量对比 ──
T = 200
np.random.seed(0)
grads = np.abs(np.random.randn(T)) + 0.5  # 模拟梯度大小

gamma = 0.9
s_adagrad, s_rmsprop = 0, 0
hist_ada, hist_rms = [], []
for g in grads:
    s_adagrad += g**2
    s_rmsprop  = gamma * s_rmsprop + (1 - gamma) * g**2
    hist_ada.append(s_adagrad)
    hist_rms.append(s_rmsprop)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_ada, label='AdaGrad s_t（无界增长）')
axes[0].plot(hist_rms, label='RMSProp s_t（有界）')
axes[0].set_title('状态量 $s_t$ 对比'); axes[0].legend()
axes[0].set_xlabel('步数')

eps = 1e-6
lr = 0.1
axes[1].plot([lr/np.sqrt(s+eps) for s in hist_ada], label='AdaGrad 有效LR')
axes[1].plot([lr/np.sqrt(s+eps) for s in hist_rms], label='RMSProp 有效LR')
axes[1].set_title('有效学习率对比'); axes[1].legend()
axes[1].set_xlabel('步数')
plt.tight_layout(); plt.show()


In [ ]:
# ── PyTorch RMSprop ──
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size=256)
net = get_net()
optimizer = torch.optim.RMSprop(net.parameters(), lr=0.01, alpha=0.9)
d2l.train_ch6(net, train_iter, test_iter, num_epochs=10, lr=0.01, device=device)


## 9. Adadelta

### 9.1 核心思想

AdaGrad 的改进变体，**无需显式设置学习率**，通过参数变化量的历史校准步长。维护两个状态变量：

**状态1 — 梯度平方的泄漏均值：**
$$\mathbf{s}_t = \rho \mathbf{s}_{t-1} + (1-\rho) \mathbf{g}_t^2$$

**重缩放梯度（利用参数变化量历史）：**
$$\mathbf{g}_t' = \frac{\sqrt{\Delta\mathbf{x}_{t-1} + \epsilon}}{\sqrt{\mathbf{s}_t + \epsilon}} \odot \mathbf{g}_t$$

**参数更新：**
$$\mathbf{x}_t = \mathbf{x}_{t-1} - \mathbf{g}_t'$$

**状态2 — 参数变化量平方的泄漏均值：**
$$\Delta\mathbf{x}_t = \rho \Delta\mathbf{x}_{t-1} + (1-\rho)(\mathbf{g}_t')^2$$

- $\rho = 0.9$（默认）
- **自校准**：分子用参数变化历史，分母用梯度历史，量纲匹配

### 9.2 PyTorch API

```python
optimizer = torch.optim.Adadelta(params, rho=0.9)
```


In [ ]:
# ── PyTorch Adadelta ──
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size=256)
net = get_net()
# Adadelta 无需设置 lr（内部自校准）
optimizer = torch.optim.Adadelta(net.parameters(), rho=0.9)
d2l.train_ch6(net, train_iter, test_iter, num_epochs=10, lr=1.0, device=device)


## 10. Adam

### 10.1 核心思想

Adam = **动量**（一阶矩）+ **自适应学习率**（二阶矩，类似 RMSProp）+ **偏差修正**（修正零初始化偏差）

### 10.2 更新规则（四步）

**① 一阶矩（动量）：**
$$\mathbf{v}_t \leftarrow \beta_1 \mathbf{v}_{t-1} + (1 - \beta_1) \mathbf{g}_t$$

**② 二阶矩（梯度平方 EMA）：**
$$\mathbf{s}_t \leftarrow \beta_2 \mathbf{s}_{t-1} + (1 - \beta_2) \mathbf{g}_t^2$$

**③ 偏差修正：**
$$\hat{\mathbf{v}}_t = \frac{\mathbf{v}_t}{1 - \beta_1^t}, \quad \hat{\mathbf{s}}_t = \frac{\mathbf{s}_t}{1 - \beta_2^t}$$

**④ 参数更新：**
$$\mathbf{x}_t \leftarrow \mathbf{x}_{t-1} - \eta \frac{\hat{\mathbf{v}}_t}{\sqrt{\hat{\mathbf{s}}_t} + \epsilon}$$

**默认超参数：** $\beta_1=0.9,\ \beta_2=0.999,\ \epsilon=10^{-6}$

### 10.3 偏差修正的必要性

初始时 $\mathbf{v}_0 = \mathbf{s}_0 = \mathbf{0}$，早期迭代中动量被低估。除以 $(1-\beta^t)$ 可修正该偏差。

### 10.4 PyTorch API

```python
optimizer = torch.optim.Adam(params, lr=0.01, betas=(0.9, 0.999), eps=1e-6)
```


In [ ]:
# ── 偏差修正效果可视化 ──
beta1, beta2 = 0.9, 0.999
T = 100
v, s = 0, 0
v_hat_list, v_raw_list = [], []
np.random.seed(1)
for t in range(1, T+1):
    g = np.random.randn()
    v = beta1 * v + (1 - beta1) * g
    s = beta2 * s + (1 - beta2) * g**2
    v_hat = v / (1 - beta1**t)
    v_hat_list.append(v_hat)
    v_raw_list.append(v)

plt.figure(figsize=(8, 4))
plt.plot(v_hat_list, label='$\hat{v}_t$（偏差修正后）')
plt.plot(v_raw_list, label='$v_t$（原始，初期偏小）', linestyle='--')
plt.xlabel('迭代步数 t'); plt.ylabel('一阶矩估计')
plt.title('Adam 偏差修正的作用（前50步尤其明显）')
plt.legend(); plt.xlim(0, 50); plt.tight_layout(); plt.show()


In [ ]:
# ── PyTorch Adam ──
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size=256)
net = get_net()
optimizer = torch.optim.Adam(net.parameters(), lr=0.001,
                              betas=(0.9, 0.999), eps=1e-6)
d2l.train_ch6(net, train_iter, test_iter, num_epochs=10, lr=0.001, device=device)


## 11. 学习率调度器

### 11.1 常用调度策略

| 策略 | 公式 | 特点 |
|---|---|---|
| 平方根衰减 | $\eta(t) = \eta_0 (t+1)^{-0.5}$ | 理论保证 |
| 指数衰减 | $\eta(t) = \eta_0 \alpha^t$ | 简单，衰减快 |
| 分段常数（多步） | 在里程碑处乘以 $\gamma$ | 实践效果好 |
| 余弦退火 | $\eta_t = \eta_T + \frac{\eta_0 - \eta_T}{2}(1+\cos\frac{\pi t}{T})$ | CV 常用 |
| 线性预热 | 初始 $k$ 步从 0 线性增加到 $\eta_0$ | 防止深层网络早期发散 |

### 11.2 PyTorch 调度器 API

```python
# 分段衰减（最常用）
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=[15, 30], gamma=0.5)

# 余弦退火
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=50, eta_min=1e-5)

# 指数衰减
scheduler = torch.optim.lr_scheduler.ExponentialLR(
    optimizer, gamma=0.95)

# 每 epoch 结束调用
scheduler.step()
```


In [ ]:
# ── 各调度策略 LR 曲线对比 ──
T = 50
t = np.arange(T)
eta0 = 0.3

strategies = {
    '常数':            np.full(T, eta0),
    '平方根衰减':       eta0 * (t + 1) ** (-0.5),
    '指数衰减 α=0.92': eta0 * (0.92 ** t),
    '多步（15,30,γ=0.5）': np.where(t < 15, eta0, np.where(t < 30, eta0*0.5, eta0*0.25)),
    '余弦退火':        1e-4 + (eta0 - 1e-4) / 2 * (1 + np.cos(np.pi * t / T)),
}

plt.figure(figsize=(9, 4))
for label, lr in strategies.items():
    plt.plot(t, lr, label=label, linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('学习率')
plt.title('常用学习率调度策略对比')
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── 实验：不同调度策略对训练的影响 ──
def train_with_scheduler(sched_fn, n_epochs=20):
    train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size=256)
    net = get_net()
    optimizer = torch.optim.SGD(net.parameters(), lr=0.3)
    scheduler = sched_fn(optimizer)
    lr_history, loss_history = [], []

    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(n_epochs):
        net.train()
        total_loss = 0
        for X, y in train_iter:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            l = loss_fn(net(X), y)
            l.backward()
            optimizer.step()
            total_loss += l.item()
        scheduler.step()
        lr_history.append(optimizer.param_groups[0]['lr'])
        loss_history.append(total_loss / len(train_iter))
    return lr_history, loss_history

# 三种调度器
sched_configs = {
    '常数': lambda opt: torch.optim.lr_scheduler.LambdaLR(opt, lambda e: 1.0),
    '多步（10,15）': lambda opt: torch.optim.lr_scheduler.MultiStepLR(opt, [10, 15], gamma=0.5),
    '余弦退火': lambda opt: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20, eta_min=1e-4),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, sched_fn in sched_configs.items():
    lr_hist, loss_hist = train_with_scheduler(sched_fn)
    axes[0].plot(lr_hist,   label=name)
    axes[1].plot(loss_hist, label=name)
axes[0].set_title('学习率变化'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].set_title('训练损失');   axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout(); plt.show()


## 12. 综合对比与实验

### 12.1 所有优化器更新规则汇总

| 优化器 | 状态变量 | 更新规则（核心） | 特点 |
|---|---|---|---|
| GD / SGD | 无 | $\mathbf{x} \leftarrow \mathbf{x} - \eta \mathbf{g}$ | 最简单，需调 LR |
| Momentum | $\mathbf{v}$ | $\mathbf{v}=\beta\mathbf{v}+\mathbf{g}$；$\mathbf{x}-=\eta\mathbf{v}$ | 加速收敛，减弱振荡 |
| AdaGrad | $\mathbf{s}$ | $\mathbf{s}+=\mathbf{g}^2$；$\mathbf{x}-=\frac{\eta}{\sqrt{\mathbf{s}+\epsilon}}\mathbf{g}$ | 自适应 LR，LR 单调降 |
| RMSProp | $\mathbf{s}$ | $\mathbf{s}=\gamma\mathbf{s}+(1-\gamma)\mathbf{g}^2$；$\mathbf{x}-=\frac{\eta}{\sqrt{\mathbf{s}+\epsilon}}\mathbf{g}$ | 修复 AdaGrad |
| Adadelta | $\mathbf{s}, \Delta\mathbf{x}$ | 见第9节 | 无 LR 参数 |
| **Adam** | $\mathbf{v}, \mathbf{s}$ | 一阶矩+二阶矩+偏差修正 | **最广泛使用** |

### 12.2 实践建议

- **首选 Adam**：收敛快，对超参不敏感，lr=1e-3/1e-4 通常可用
- **SGD + Momentum + LR调度**：精调时可超越 Adam（尤其视觉任务）
- **AdaGrad/RMSProp**：NLP/稀疏特征场景
- **学习率调度必不可少**：即使用 Adam 也应配合 warmup 或余弦退火


In [ ]:
# ── 全优化器横向对比实验 ──
def train_optimizer(optimizer_fn, n_epochs=15):
    train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size=256)
    net = get_net()
    optimizer = optimizer_fn(net.parameters())
    loss_fn = nn.CrossEntropyLoss()
    loss_history = []
    for epoch in range(n_epochs):
        net.train()
        total_loss = 0
        for X, y in train_iter:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            l = loss_fn(net(X), y)
            l.backward()
            optimizer.step()
            total_loss += l.item()
        loss_history.append(total_loss / len(train_iter))
    return loss_history

optimizer_configs = {
    'SGD (lr=0.1)':          lambda p: torch.optim.SGD(p, lr=0.1),
    'SGD+Momentum (β=0.9)':  lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9),
    'Adagrad':               lambda p: torch.optim.Adagrad(p, lr=0.1),
    'RMSprop':               lambda p: torch.optim.RMSprop(p, lr=0.01, alpha=0.9),
    'Adadelta':              lambda p: torch.optim.Adadelta(p, rho=0.9),
    'Adam':                  lambda p: torch.optim.Adam(p, lr=0.001),
}

plt.figure(figsize=(10, 5))
for name, opt_fn in optimizer_configs.items():
    losses = train_optimizer(opt_fn)
    plt.plot(losses, label=name, linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('训练损失（CE）')
plt.title('各优化器收敛曲线对比（LeNet on Fashion-MNIST）')
plt.legend(loc='upper right'); plt.tight_layout(); plt.show()


In [ ]:
# ── PyTorch 优化器 API 速查 ──
print("=" * 55)
print("PyTorch 优化器 API 速查")
print("=" * 55)
apis = [
    ("SGD",       "torch.optim.SGD(params, lr=0.01)"),
    ("Momentum",  "torch.optim.SGD(params, lr=0.005, momentum=0.9)"),
    ("Adagrad",   "torch.optim.Adagrad(params, lr=0.1)"),
    ("RMSprop",   "torch.optim.RMSprop(params, lr=0.01, alpha=0.9)"),
    ("Adadelta",  "torch.optim.Adadelta(params, rho=0.9)"),
    ("Adam",      "torch.optim.Adam(params, lr=1e-3, betas=(0.9,0.999))"),
    ("MultiStep", "lr_scheduler.MultiStepLR(opt, milestones=[15,30], gamma=0.5)"),
    ("Cosine",    "lr_scheduler.CosineAnnealingLR(opt, T_max=50)"),
    ("Exponential","lr_scheduler.ExponentialLR(opt, gamma=0.95)"),
]
for name, api in apis:
    print(f"  {name:<12}  {api}")
